# Final Project by Denise Godinez (00001614313), Jack Mulvihill (00001623740), and Fareen Samad (00001663164)

## Abstract

## Imported Libraries

In [2]:
import glob

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Perceptron
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

## Data Extraction

In [3]:
# Preview one of the data sources
ex = pd.read_json('ids_4.json', lines=True)

# Display the columns
print(ex.columns)

display(ex.head())

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,27887,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,181,2,2,74,154,37,37,37.0,0.0,...,44,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,80,115,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,13,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,56529,50,1,1,0,0,0,0,0.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
"""This functions combines all the csv, json, and parquet files into a dataframe

    Args:
        None: The function reads all csv, json, and parquet files in the working directory
        
    Returns:
        data (pd.DataFrame): All data sources combined in a single dataframe
    """
def extract() -> pd.DataFrame:
    
    # Create a new dataframe to hold all the data
    # Column names chosen based on what was returned from .columns
    data = pd.DataFrame(columns=[' Destination Port', ' Flow Duration', ' Total Fwd Packets',
                                 ' Total Backward Packets', 'Total Length of Fwd Packets',
                                 ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
                                 ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
                                 ' Fwd Packet Length Std', 'Bwd Packet Length Max',
                                 ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
                                 ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
                                 ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
                                 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
                                 ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
                                 ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
                                 ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
                                 ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
                                 ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
                                 ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count',
                                 ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count',
                                 ' ACK Flag Count', ' URG Flag Count', ' CWE Flag Count',
                                 ' ECE Flag Count', ' Down/Up Ratio', ' Average Packet Size',
                                 ' Avg Fwd Segment Size', ' Avg Bwd Segment Size',
                                 ' Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk',
                                 ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk',
                                 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', ' Subflow Fwd Bytes',
                                 ' Subflow Bwd Packets', ' Subflow Bwd Bytes', 'Init_Win_bytes_forward',
                                 ' Init_Win_bytes_backward', ' act_data_pkt_fwd',
                                 ' min_seg_size_forward', 'Active Mean', ' Active Std', ' Active Max',
                                 ' Active Min', 'Idle Mean', ' Idle Std', ' Idle Max', ' Idle Min',' Label'])
    
    # Load the csv files
    for csvfile in glob.glob('*.csv'):
        
        # Load current csv into the temp dataframe
        temp = pd.read_csv(csvfile)
        
        # Combine the loaded csv files into data
        data = pd.concat([data, temp], ignore_index=True)
    
    # Load the json files
    for jsonfile in glob.glob('*.json'):
        
        # Load current json into the temp dataframe
        temp = pd.read_json(jsonfile, lines=True)
        
        # Combine the loaded json files into data
        data = pd.concat([data, temp], ignore_index=True)
    
    # Load the parquet files
    for parquetfile in glob.glob('*.parquet'):
        
        # Load the current parquet file into the temp dataframe
        temp = pd.read_parquet(parquetfile)
        
        # Combine the loaded parquet files into data
        data = pd.concat([data, temp], ignore_index=True)

    # Standardize all column names to lowercase with underscores before deduplicating
    data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('/', '_')
    
    # Keep only the first occurrence of each duplicate column name
    data = data.loc[:, ~data.columns.duplicated()]

    # Drop all rows that are entirely NaN
    data = data.dropna(how='all')

    # drop rows with no label
    data = data.dropna(subset=['label']) 

    # Return the newly extracted data
    return data

Because all of these data sources share the same attributes, the best approach is to combine all the data from all the files into one combined DataFrame for easy analysis.

In [5]:
# Extract the data into a dataframe data
data = extract()

# Display all columns
pd.set_option('display.max_columns', None)
display(data.head())

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,cwe_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,fwd_header_length.1,fwd_avg_bytes_bulk,fwd_avg_packets_bulk,fwd_avg_bulk_rate,bwd_avg_bytes_bulk,bwd_avg_packets_bulk,bwd_avg_bulk_rate,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,53,61205,4,2,136,428,34,34,34.0,0.0,214,214,214.0,0.0,9214.93342,98.031207,12241.0,16867.17138,33318,3,33324,11108.0,19234.42422,33318,3,4,4.0,0.0,4,4,0,0,0,0,128,40,65.354138,32.677069,34,214,85.428571,87.831007,7714.285714,0,0,0,0,0,0,0,0,0,99.666667,34.0,214.0,128,0,0,0,0,0,0,4,136,2,428,-1,-1,3,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53,222,2,2,90,172,45,45,45.0,0.0,86,86,86.0,0.0,1180180.18,18018.01802,74.0,86.0,170,4,48,48.0,0.0,48,48,4,4.0,0.0,4,4,0,0,0,0,40,40,9009.009009,9009.009009,45,86,61.4,22.456625,504.3,0,0,0,0,0,0,0,0,1,76.75,45.0,86.0,40,0,0,0,0,0,0,2,90,2,172,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,53,23759,2,2,70,126,35,35,35.0,0.0,63,63,63.0,0.0,8249.505451,168.357254,7919.666667,13711.20288,23752,3,4,4.0,0.0,4,4,3,3.0,0.0,3,3,0,0,0,0,64,40,84.178627,84.178627,35,63,46.2,15.336232,235.2,0,0,0,0,0,0,0,0,1,57.75,35.0,63.0,64,0,0,0,0,0,0,2,70,2,126,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,80,401,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,29925.18703,4987.531172,401.0,0.0,401,401,401,401.0,0.0,401,401,0,0.0,0.0,0,0,0,0,0,0,40,0,4987.531172,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,253,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,57406,4,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,3000000.0,500000.0,4.0,0.0,4,4,4,4.0,0.0,4,4,0,0.0,0.0,0,0,0,0,0,0,40,0,500000.0,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,362,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,N

## Data Transformation

In [6]:
# Display the data types of each attribute in the newly created combined DataFrame
print(data.dtypes)

destination_port                object
flow_duration                   object
total_fwd_packets               object
total_backward_packets          object
total_length_of_fwd_packets     object
                                ...   
62                             float64
63                             float64
64                             float64
65                             float64
66                             float64
Length: 146, dtype: object


All of these data types seem to be object. So, convert them as necessary based on the information provided when displaying data.

In [7]:
def transform(data: pd.DataFrame) -> pd.DataFrame:
    
    # Drop any remaining duplicate columns
    data = data.loc[:, ~data.columns.duplicated()]
    
    # Drop rows where label is missing
    data = data.dropna(subset=['label'])
    
    # Convert all non-label columns to numeric, coercing errors to NaN
    for col in data.columns:
        if col != 'label':
            data[col] = pd.to_numeric(data[col], errors='coerce')
    

   # Drop columns that are entirely NaN
    data = data.dropna(axis=1, how='all')

    # Drop the .1 duplicate columns that slipped through, but keep label
    data = data[[c for c in data.columns if not c.endswith('.1') or c == 'label']]

    # Drop rows where label is missing
    data = data.dropna(subset=['label'])
    
    # Ensure label is a string
    data['label'] = data['label'].astype('string')

    # Drop the small number of remaining NaN rows
    data = data.dropna()
    
    return data

In [38]:
# Transform the data and display the first few rows to verify the changes
data = transform(data)
display(data.head())

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,cwe_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,fwd_avg_bytes_bulk,fwd_avg_packets_bulk,fwd_avg_bulk_rate,bwd_avg_bytes_bulk,bwd_avg_packets_bulk,bwd_avg_bulk_rate,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
110264,55109,17,1,1,6,6,6,6,6.0,0.000000,6,6,6.0,0.000000,705882.352900,117647.058800,17.0,0.0,17,17,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,58823.529410,58823.529410,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000,6.0,6.0,0,0,0,0,0,0,1,6,1,6,1013,16213,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
110265,53,113594958,4,4,152,362,45,31,38.0,8.082904,120,61,90.5,34.063666,4.524849,0.070426,16200000.0,42900000.0,114000000,3,114000000,37900000.0,65600000.0,114000000,3,114000000,37900000.0,65600000.0,114000000,48,0,0,0,0,128,128,0.035213,0.035213,31,120,60.555556,35.658488,1271.527778,0,0,0,0,0,0,0,0,1,68.125,38.0,90.5,0,0,0,0,0,0,4,152,4,362,-1,-1,3,32,240.0,0.0,240,240,114000000.0,0.0,114000000,114000000,BENIGN
110266,53,30485,1,1,81,209,81,81,81.0,0.000000,209,209,209.0,0.000000,9512.875185,65.606036,30485.0,0.0,30485,30485,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,32,32,32.803018,32.803018,81,209,123.666667,73.900834,5461.333333,0,0,0,0,0,0,0,0,1,185.500,81.0,209.0,0,0,0,0,0,0,1,81,1,209,-1,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
110267,53,30445,1,1,53,81,53,53,53.0,0.000000,81,81,81.0,0.000000,4401.379537,65.692232,30445.0,0.0,30445,30445,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,32,20,32.846116,32.846116,53,81,62.333333,16.165808,261.333333,0,0,0,0,0,0,0,0,1,93.500,53.0,81.0,0,0,0,0,0,0,1,53,1,81,-1,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
110268,53,70860,1,1,56,72,56,56,56.0,0.000000,72,72,72.0,0.000000,1806.378775,28.224668,70860.0,0.0,70860,70860,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,32,32,14.112334,14.112334,56,72,61.333333,9.237604,85.333333,0,0,0,0,0,0,0,0,1,92.000,56.0,72.0,0,0,0,0,0,0,1,56,1,72,-1,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [8]:
# Check the shape of the data and look for any remaining duplicate columns
print(data.shape)
print([c for c in data.columns if data.columns.tolist().count(c) > 1])

(63129, 146)
[]


The naming of the columns was inconsistent and contained a lot of unnecessary whitespace, which made it confusing for the users. So, the main goal of transformation was to convert all of the feature names into a standardized format to make working with them easier down the line. In doing so, a duplicate column was identified and deleted from the dataset. The other major issue that needed to be addressed was that all of the data types were of type "object." Based on what was provided when previewing the data, converted all of the data types as appropriate. This included changing the counts and numbers to integers, statistical data to floats, and labels to strings.  

## Data Loading and Storage

In [9]:
"""This function loads the argument dataframe data into the csv file identified by filename

    Args:
        data (pd.DataFrame): transformed dataframe
        filename (str): the csv
    """
def loadcsv(data: pd.DataFrame, filename: str) -> None:
    
    # Use .to_csv to convert the dataframe into a CSV
    # use an f-string to write the filename as the string parameter, do not index
    data.to_csv(f'{filename}', index=False)

In [10]:
# Create the CSV file with the transformed data
loadcsv(data, 'transformed_data.csv')

Loading the transformed data into a CSV file seems like the most logical choice here given that no use of SQL will be necessary in the foreseeable future.

## Data Reading

In [11]:
# Load the transformed data into a new dataframe to confirm the transformations were successful
cleaned_data = pd.read_csv('transformed_data.csv')

# Display all columns
pd.set_option('display.max_columns', None)
display(cleaned_data.head())

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,cwe_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,fwd_header_length.1,fwd_avg_bytes_bulk,fwd_avg_packets_bulk,fwd_avg_bulk_rate,bwd_avg_bytes_bulk,bwd_avg_packets_bulk,bwd_avg_bulk_rate,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,53,61205,4,2,136,428,34,34,34.0,0.0,214,214,214.0,0.0,9.214933e+03,98.031207,12241.000000,16867.17138,33318,3,33324,11108.0,19234.42422,33318,3,4,4.0,0.0,4,4,0,0,0,0,128,40,65.354138,32.677069,34,214,85.428571,87.831007,7714.285714,0,0,0,0,0,0,0,0,0,99.666667,34.0,214.0,128,0,0,0,0,0,0,4,136,2,428,-1,-1,3,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53,222,2,2,90,172,45,45,45.0,0.0,86,86,86.0,0.0,1.180180e+06,18018.018020,74.000000,86.00000,170,4,48,48.0,0.00000,48,48,4,4.0,0.0,4,4,0,0,0,0,40,40,9009.009009,9009.009009,45,86,61.400000,22.456625,504.300000,0,0,0,0,0,0,0,0,1,76.750000,45.0,86.0,40,0,0,0,0,0,0,2,90,2,172,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,53,23759,2,2,70,126,35,35,35.0,0.0,63,63,63.0,0.0,8.249505e+03,168.357254,7919.666667,13711.20288,23752,3,4,4.0,0.00000,4,4,3,3.0,0.0,3,3,0,0,0,0,64,40,84.178627,84.178627,35,63,46.200000,15.336232,235.200000,0,0,0,0,0,0,0,0,1,57.750000,35.0,63.0,64,0,0,0,0,0,0,2,70,2,126,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,80,401,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,2.992519e+04,4987.531172,401.000000,0.00000,401,401,401,401.0,0.00000,401,401,0,0.0,0.0,0,0,0,0,0,0,40,0,4987.531172,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.000000,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,253,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,57406,4,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,3.000000e+06,500000.000000,4.000000,0.00000,4,4,4,4.0,0.00000,4,4,0,0.0,0.0,0,0,0,0,0,0,40,0,500000.0000

Data cleaning steps were successful, load into a new DataFrame.

## Data Analysis

In [12]:
# Display the dimensions of the newly transformed data
cleaned_data.shape

(63129, 146)

In [13]:
# Identify unique values in the label column and their counts
unique_labels = cleaned_data['label'].value_counts()
print(unique_labels)

label
DoS Hulk            31027
DoS GoldenEye       20586
BENIGN               6006
DoS Slowhttptest     5499
Heartbleed             11
Name: count, dtype: int64


In [14]:
# Identify if the dataset has missing data
cleaned_data.isna().sum()

destination_port                   0
flow_duration                      0
total_fwd_packets                  0
total_backward_packets             0
total_length_of_fwd_packets        0
                               ...  
62                             63129
63                             63129
64                             63129
65                             63129
66                             63129
Length: 146, dtype: int64

There does not seem to be any missing data.

In [15]:
# Display the summary statistics of the transformed data
cleaned_data.describe()

c:\Users\oreof\anaconda3\envs\comp379\Lib\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\oreof\anaconda3\envs\comp379\Lib\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,cwe_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,fwd_header_length.1,fwd_avg_bytes_bulk,fwd_avg_packets_bulk,fwd_avg_bulk_rate,bwd_avg_bytes_bulk,bwd_avg_packets_bulk,bwd_avg_bulk_rate,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
count,63129.000000,6.312900e+04,63129.000000,63129.000000,63129.000000,6.312900e+04,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,6.301100e+04,6.309600e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,63129.000000,63129.0,63129.0,63129.0,63129.000000,63129.000000,6.312900e+04,6.312900e+04,63129.000000,63129.000000,63129.000000,63129.000000,6.312900e+04,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.0,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.000000,63129.0,63129.0,63129.0,63129.0,63129.0,63129.0,63129.000000,63129.000000,63129.000000,6.312900e+04,63129.000000,63129.000000,63129.000000,63129.000000,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,6.312900e+04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,995.459282,4.184000e+07,6.851637,5.363478,403.101697,1.010193e+04,261.484009,12.284845,61.615402,101.822935,3393.677359,5.014003,1060.907669,1461.395392,inf,inf,7.779665e+06,1.084627e+07,3.815589e+07,3.787252e+06,4.063093e+07,1.135878e+07,1.380382e+07,3.798113e+07,5.124604e+06,1.483796e+07,3.445837e+06,5.933046e+06,1.300404e+07,2.530640e+05,0.019848,0.0,0.0,0.0,210.455987,160.382582,9.466179e+04,1.881369e+03,3.646771,3427.896007,477.792807,1041.859285,2.007327e+06,0.127833,0.019848,0.000016,0.354829,0.476263,0.023539,0.0,0.000016,0.293193,520.589252,61.615402,1060.907669,210.455987,0.0,0.0,0.0,0.0,0.0,0.0,6.851637,403.101697,5.363478,1.010119e+04,10417.966165,526.269163,2.168274,29.579179,4.161182e+05,3.248900e+04,4.444066e+05,3.910737e+05,3.668389e+07,1.030739e+06,3.768315e+07,3.585323e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,6709.013519,4.278803e+07,112.410345,147.094619,2319.363366,3.383084e+05,323.336961,124.842995,143.700401,120.841347,3311.850503,28.161288,973.769125,1463.261835,NaN,NaN,1.611655e+07,1.418

Because there are so many features, including duplicates, generating visuals such as pair plots would be unwieldly and difficult to read. 

## Data Preprocessing

In [16]:
# Display the cleaned data to confirm the transformations were successful
display(cleaned_data.head())

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,cwe_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,fwd_header_length.1,fwd_avg_bytes_bulk,fwd_avg_packets_bulk,fwd_avg_bulk_rate,bwd_avg_bytes_bulk,bwd_avg_packets_bulk,bwd_avg_bulk_rate,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,53,61205,4,2,136,428,34,34,34.0,0.0,214,214,214.0,0.0,9.214933e+03,98.031207,12241.000000,16867.17138,33318,3,33324,11108.0,19234.42422,33318,3,4,4.0,0.0,4,4,0,0,0,0,128,40,65.354138,32.677069,34,214,85.428571,87.831007,7714.285714,0,0,0,0,0,0,0,0,0,99.666667,34.0,214.0,128,0,0,0,0,0,0,4,136,2,428,-1,-1,3,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53,222,2,2,90,172,45,45,45.0,0.0,86,86,86.0,0.0,1.180180e+06,18018.018020,74.000000,86.00000,170,4,48,48.0,0.00000,48,48,4,4.0,0.0,4,4,0,0,0,0,40,40,9009.009009,9009.009009,45,86,61.400000,22.456625,504.300000,0,0,0,0,0,0,0,0,1,76.750000,45.0,86.0,40,0,0,0,0,0,0,2,90,2,172,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,53,23759,2,2,70,126,35,35,35.0,0.0,63,63,63.0,0.0,8.249505e+03,168.357254,7919.666667,13711.20288,23752,3,4,4.0,0.00000,4,4,3,3.0,0.0,3,3,0,0,0,0,64,40,84.178627,84.178627,35,63,46.200000,15.336232,235.200000,0,0,0,0,0,0,0,0,1,57.750000,35.0,63.0,64,0,0,0,0,0,0,2,70,2,126,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,80,401,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,2.992519e+04,4987.531172,401.000000,0.00000,401,401,401,401.0,0.00000,401,401,0,0.0,0.0,0,0,0,0,0,0,40,0,4987.531172,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.000000,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,253,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,57406,4,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,3.000000e+06,500000.000000,4.000000,0.00000,4,4,4,4.0,0.00000,4,4,0,0.0,0.0,0,0,0,0,0,0,40,0,500000.0000

In [17]:
# Remove duplicate columns in place
cleaned_data = cleaned_data.loc[:, ~cleaned_data.columns.str.endswith('.1')]

# Identify columns with zero variance
zero_variance_cols = cleaned_data.columns[cleaned_data.nunique() <= 1]
print(zero_variance_cols)

# Drop the zero variance columns
cleaned_data = cleaned_data.drop(columns=zero_variance_cols)

Index(['bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'cwe_flag_count',
       'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate',
       'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate', '0',
       '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13',
       '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37',
       '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49',
       '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61',
       '62', '63', '64', '65', '66'],
      dtype='str')


In [18]:
# Display the first few rows of the cleaned data to confirm the desired columns were removed
display(cleaned_data.head())

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,bwd_packet_length_min,bwd_packet_length_mean,bwd_packet_length_std,flow_bytes_s,flow_packets_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_total,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_total,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,fwd_header_length,bwd_header_length,fwd_packets_s,bwd_packets_s,min_packet_length,max_packet_length,packet_length_mean,packet_length_std,packet_length_variance,fin_flag_count,syn_flag_count,rst_flag_count,psh_flag_count,ack_flag_count,urg_flag_count,ece_flag_count,down_up_ratio,average_packet_size,avg_fwd_segment_size,avg_bwd_segment_size,subflow_fwd_packets,subflow_fwd_bytes,subflow_bwd_packets,subflow_bwd_bytes,init_win_bytes_forward,init_win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,53,61205,4,2,136,428,34,34,34.0,0.0,214,214,214.0,0.0,9.214933e+03,98.031207,12241.000000,16867.17138,33318,3,33324,11108.0,19234.42422,33318,3,4,4.0,0.0,4,4,0,128,40,65.354138,32.677069,34,214,85.428571,87.831007,7714.285714,0,0,0,0,0,0,0,0,99.666667,34.0,214.0,4,136,2,428,-1,-1,3,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,222,2,2,90,172,45,45,45.0,0.0,86,86,86.0,0.0,1.180180e+06,18018.018020,74.000000,86.00000,170,4,48,48.0,0.00000,48,48,4,4.0,0.0,4,4,0,40,40,9009.009009,9009.009009,45,86,61.400000,22.456625,504.300000,0,0,0,0,0,0,0,1,76.750000,45.0,86.0,2,90,2,172,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,53,23759,2,2,70,126,35,35,35.0,0.0,63,63,63.0,0.0,8.249505e+03,168.357254,7919.666667,13711.20288,23752,3,4,4.0,0.00000,4,4,3,3.0,0.0,3,3,0,64,40,84.178627,84.178627,35,63,46.200000,15.336232,235.200000,0,0,0,0,0,0,0,1,57.750000,35.0,63.0,2,70,2,126,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,401,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,2.992519e+04,4987.531172,401.000000,0.00000,401,401,401,401.0,0.00000,401,401,0,0.0,0.0,0,0,0,40,0,4987.531172,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,9.000000,6.0,0.0,2,12,0,0,253,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,57406,4,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,3.000000e+06,500000.000000,4.000000,0.00000,4,4,4,4.0,0.00000,4,4,0,0.0,0.0,0,0,0,40,0,500000.000000,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,9.000000,6.0,0.0,2,12,0,0,362,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [19]:
# Save the output of the network traffic as the "labels" for the classifier
y = cleaned_data.pop('label')

# Convert to numpy array to one-hot encode the data
y = y.to_numpy()
y = y.squeeze()

In [20]:
print(y)

# Check the shape of the resulting variable target
y.shape

['BENIGN' 'BENIGN' 'BENIGN' ... 'DoS GoldenEye' 'DoS GoldenEye'
 'DoS GoldenEye']


(63129,)

In [21]:
# Use .unique() with return_counts=True to get the unique values
# of the target along with their frequencies
unique_val, freq = np.unique(y, return_counts=True)
print(f'Unique values: {unique_val}\nFrequencies: {freq}')

Unique values: ['BENIGN' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest' 'Heartbleed']
Frequencies: [ 6006 20586 31027  5499    11]


The dataset seems to be pretty unbalanced, with most of the samples being malicious traffic.

In [22]:
# Encode the label column 0 or 1 depending on if the label is benign or malicious
# 0 - BENIGN
# 1 - MALICIOUS: DoS Hulk, DoS GoldenEye, Heartbleed, DoS Slowhttptest
y = np.where(y == 'BENIGN', 0, 1)

In [23]:
# Verify that target includes numeric values and display their frequencies

# Get the unique values of the target along with their frequencies
unique_val, freq = np.unique(y, return_counts=True)
print(f'Unique values: {unique_val}\nFrequencies: {freq}')

Unique values: [0 1]
Frequencies: [ 6006 57123]


In [34]:
# Split the data into training and test datasets
# Set the test_size to 0.15 to construct a test dataset from 15% of the data
# Stratify the target to ensure the proportion between the different classes of the dataset
# in the training and test datasets is the same as it was in the original dataset 
X_train, X_test, y_train, y_test = train_test_split(cleaned_data, y, random_state=1, stratify=y)

In [35]:
# Display the sizes of the resulting four variables

# Use .shape to get the dimensions of the datasets, 0 for rows and 1 for columns
print(f"There are {X_train.shape[0]} rows and {X_train.shape[1]} columns in X_train")
print(f"There are {X_test.shape[0]} rows and {X_test.shape[1]} columns in X_test")
print(f"There are {y_train.shape[0]} rows and 1 column in y_train")
print(f"There are {y_test.shape[0]} rows and 1 column in y_test")

There are 47346 rows and 67 columns in X_train
There are 15783 rows and 67 columns in X_test
There are 47346 rows and 1 column in y_train
There are 15783 rows and 1 column in y_test


## Feature Engineering

In [ ]:
# There were some infinite values in the data that were causing issues with the standardization
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Drop the rows with infinite values in the training and test datasets
X_train.dropna(inplace=True)
X_test.dropna(inplace=True)

# TODO: Drop the corresponding labels in y at the same index in X that had the NaN values

IndexError: index 52837 is out of bounds for axis 0 with size 47346

In [27]:
# Create a standardizing object (make the mean=0 and variance=1)
scaler = StandardScaler()

# Use the training dataset to fit the standardizing object
scaler.fit(X_train)

# Standardize the training dataset
X_train = scaler.transform(X_train)

# Standardize the test dataset
X_test  = scaler.transform(X_test)

In [28]:
# Verify the training data is standardized 

# Create a temporary dataframe to display the statistical information of X_train
temp = pd.DataFrame(X_train).describe()
display(temp)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
count,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,47257.000000,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04,4.725700e+04
mean,3.307857e-17,6.074428e-17,4.360357e-18,3.007143e-18,-4.811428e-18,-6.014286e-19,-3.548429e-17,-1.187821e-17,4.961786e-18,1.398321e-17,-5.623357e-17,-2.646286e-17,2.706429e-18,-1.482521e-16,0.000000,1.202857e-17,6.976571e-17,4.811428e-17,-5.803786e-17,-6.014286e-18,-3.849143e-17,4.360357e-18,1.563714e-17,-1.578750e-17,-6.615714e-18,-5.052000e-17,1.638893e-17,-6.014286e-19,-3.578500e-17,-5.112143e-18,1.563714e-17,2.105000e-18,3.007143e-18,-3.127429e-17,-7.668214e-18,1.052500e-17,3.007143e-18,1.383286e-17,4.375393e-17,1.175793e-16,7.262250e-17,1.563714e-17,-6.014286e-19,-6.435286e-17,1.296079e-16,-3.608571e-18,-6.014286e-19,1.323143e-17,1.933593e-16,4.961786e-18,2.706429e-18,4.360357e-18,-4.811428e-18,3.007143e-18,3.007143e-18,-5.803786e-17,3.608571e-18,1.202857e-18,-1.215637e-16,-3.683750e-17,2.435786e-17,2.886857e-17,-3.608571e-17,8.510214e-17,-2.255357e-18,-2.766571e-17,2.706429e-18
std,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00,1.000011e+00
min,-1.476497e-01,-9.804000e-01,-4.763584e-02,-3.371248e-02,-1.573951e-01,-2.801345e-02,-8.232155e-01,-9.867193e-02,-4.335652e-01,-8.563838e-01,-1.023489e+00,-1.762037e-01,-1.088380e+00,-9.974146e-01,-0.051186,-2.893610e-01,-4.845184e-01,-7.644652e-01,-9.002566e-01,-2.433003e-01,-9.337401e-01,-5.908988e-01,-7.551738e-01,-8.931246e-01,-2.711199e-01,-4.707022e-01,-3.871096e-01,-4.071946e-01,-4.277161e-01,-7.393684e-02,-1.413740e-01,-7.888573e-02,-4.844804e-02,-2.861843e-01,-6.948183e-02,-1.429995e-01,-1.041222e+00,-1.150816e+00,-1.083718e+00,-8.308896e-01,-3.813163e-01,-1.413740e-01,-4.600145e-03,-7.427646e-01,-9.530448e-01,-1.553781e-01,-4.600145e-03,-5.570512e-01,-1.148846e+00,-4.335652e-01,-1.088380e+00,-4.763584e-02,-1.573951e-01,-3.371248e-02,-2.801799e-02,-7.486736e-01,-1.421742e-01,-2.006876e-02,-5.329393e+00,-2.676169e-01,-7.827950e-02,-2.697473e-01,-2.568888e-01,-8.597907e-01,-1.868393e-01,-8.814696e-01,-8.341166e-01
25%,-1.356804e-01,-9.769269e-01,-3.981962e-02,-3.371248e-02,-1.573951e-01,-2.801345e-02,-8.232155e-01,-9.867193e-02,-4.335652e-01,-8.563838e-01,-1.023489e+00,-1.762037e-01,-1.088380e+00,-9.974146e-01,-0.051186,-2.893606e-

In [29]:
# Verify the test data is standardized 

# Create a temporary dataframe to display the statistical information of X_test
temp = pd.DataFrame(X_test).describe()
display(temp)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
count,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,1.574900e+04,15749.000000,15749.000000,15749.000000,1.574900e+04,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000,15749.000000
mean,0.003710,-0.003767,-0.007296,-0.007744,-0.004262,-0.007115,0.013284,0.002927,0.010823,0.011464,0.008873,-0.004452,0.009734,0.007135,0.002933,0.003667,-0.013721,-0.004242,-0.002216,-0.009168,-0.003794,-0.012121,0.004392,-0.002248,-0.013367,0.012629,0.001950,0.007725,0.012919,0.000685,0.007971,-0.006200,-0.007088,0.003329,0.004555,-0.005460,0.009751,0.014169,0.010448,0.005367,0.013686,0.007971,-4.600145e-03,-0.000197,-0.005859,-0.000525,-4.600145e-03,0.012024,0.014327,0.010823,0.009734,-0.007296,-0.004262,-0.007744,-0.007114,0.000385,0.008787,-0.008424,-0.015047,-0.009343,0.004160,-0.008090,-0.011104,0.000362,-0.020690,-0.001882,0.002378
std,1.009344,0.998958,0.306125,0.178487,0.479236,0.285430,1.079270,1.036389,1.063734,1.070320,0.995862,0.927956,0.996293,0.994068,0.979587,1.000169,0.981547,0.986765,0.999434,0.983490,0.999057,0.979122,1.000983,0.999460,0.974960,1.010357,0.983336,0.997145,1.011187,1.014537,1.027256,0.461840,0.278037,0.998249,1.000296,0.952218,0.995771,1.000355,0.995697,0.993018,1.015161,1.027256,8.673893e-19,0.999972,0.999733,0.998381,8.673893e-19,1.004435,1.001552,1.063734,0.996293,0.306125,0.479236,0.178487,0.285502,1.000304,1.037037,0.038326,1.000411,0.979900,1.011715,0.984820,0.976792,1.000331,0.918605,0.999253,1.000013
min,-0.144508,-0.980400,-0.047636,-0.033712,-0.157395,-0.028013,-0.823216,-0.098672,-0.433565,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,-0.051186,-0.289361,-0.484518,-0.764465,-0.900257,-0.243300,-0.933740,-0.590899,-0.755174,-0.893125,-0.271120,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.071546,-0.048448,-0.286184,-0.069482,-0.143000,-1.041222,-1.150816,-1.083718,-0.830890,-0.381316,-0.141374,-4.600145e-03,-0.742765,-0.953045,-0.155378,-4.600145e-03,-0.557051,-1.148846,-0.433565,-1.088380,-0.047636,-0.157395,-0.033712,-0.028018,-0.748674,-0.142174,-0.020069,-1.728001,-0.267617,-0.078280,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
25%,-0.135680,-0.976885,-0.039820,-0.033712,-0.157395,-0.028013,-0.823216,-0.098672,-0.433565,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,-0.051186,-0.289361,-0.482625,-0.764465,-0.897075,-0.243300,-0.933488,-0.590641,-0.755174,-0.892891,-0.271120,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.055400,-0.048448,-0.286184,-0.069482,-0.143000,-1.041222,-1.150816,-1.083718,-0.830890,-0.381316,-0.141374,-4.600145e-03,-0.742765,-0.953045,-0.155378,-4.600145e-03,-0.557051,-1.148846,-0.433565,-1.088380,-0.039820,-0.157395,-0.033712,-0.028018,-0.730598,-0.142174,-0.020069,0.432835,-0.267617,-0.078280,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
50%,-0.135680,-0.701433,-0.008555,-0.004147,-0.027342,0.002040,0.205247,-0.098672,-0.101424,0.182360,0.039833,-0.176204,0.117388,0.200076,-0.051182,-0.289359,-0.393625,-0.604275,-0.736689,-0.243299,-0.756159,-0.500952,-0.622047,-0.730306,-0.271120,-0.466097,-0.383933,-0.403380,-0.423373,-0.073932,-0.141

## Processed Data Loading

In [30]:
# Save the standardized training and test datasets to new CSV files
pd.DataFrame(X_train).to_csv('X_train.csv', index=False)
pd.DataFrame(X_test).to_csv('X_test.csv', index=False)
pd.DataFrame(y_train).to_csv('y_train.csv', index=False)
pd.DataFrame(y_test).to_csv('y_test.csv', index=False)

# Display the first few rows of the standardized training data to confirm the transformations were successful
display(pd.DataFrame(X_train).head())
# Display the first few rows of the standardized test data to confirm the transformations were successful
display(pd.DataFrame(X_test).head())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,-0.135680,-0.863389,-0.000739,-0.004147,0.004010,0.002136,0.492334,-0.098672,-0.012584,0.471056,1.611741,-0.176204,1.298938,1.593083,-0.051128,-0.289354,-0.456419,-0.658550,-0.782557,-0.243300,-0.933583,-0.590840,-0.755102,-0.893053,-0.271120,-0.311078,-0.246554,-0.235091,-0.262242,-0.073884,-0.141374,0.006250,0.000350,-0.286180,-0.069444,-0.143000,1.613365,1.084105,1.495456,1.710838,-0.381316,-0.141374,-0.0046,1.346322,-0.953045,-0.155378,-0.0046,-0.557051,1.069565,-0.012584,1.298938,-0.000739,0.004010,-0.004147,0.002139,1.345897,-0.077680,-0.011800,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
1,8.272485,-0.980400,-0.039820,-0.033712,-0.152750,-0.028013,-0.804287,-0.050215,-0.391164,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,0.236917,5.712347,-0.484518,-0.764465,-0.900257,-0.243300,-0.933740,-0.590899,-0.755174,-0.893125,-0.271120,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.064207,-0.048448,5.766044,-0.069482,0.089607,-1.039399,-1.136348,-1.083718,-0.830890,-0.381316,-0.141374,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,-0.557051,-1.128962,-0.391164,-1.088380,-0.039820,-0.152750,-0.033712,-0.028018,-0.735691,-0.142174,-0.011800,-1.728001,-0.267617,-0.07828,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
2,-0.135680,1.326649,-0.008555,0.001767,-0.014569,0.002040,0.284117,-0.098672,0.001045,0.338361,0.723678,-0.176204,0.894724,0.616778,-0.051183,-0.289361,0.069512,1.328895,1.422521,-0.243299,1.327768,0.426680,1.656232,1.421822,-0.271120,2.675200,1.824482,2.627887,2.836763,-0.073922,-0.141374,-0.018704,0.009645,-0.286184,-0.069480,-0.143000,0.718779,1.069451,0.804009,0.530699,2.622495,-0.141374,-0.0046,-0.742765,-0.953045,-0.155378,-0.0046,1.349835,1.055020,0.001045,0.894724,-0.008555,-0.014569,0.001767,0.002043,-0.748602,-0.077680,0.004737,-1.728001,-0.259985,-0.07828,-0.262535,-0.249110,1.449258,-0.186839,1.421554,1.459274
3,-0.135680,-0.980400,-0.039820,-0.033712,-0.157395,-0.028013,-0.823216,-0.098672,-0.433565,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,-0.051186,5.712347,-0.484518,-0.764465,-0.900257,-0.243300,-0.933740,-0.590899,-0.755174,-0.893125,-0.271120,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.055400,-0.048448,5.766044,-0.069482,-0.143000,-1.041222,-1.150816,-1.083718,-0.830890,-0.381316,-0.141374,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,-0.557051,-1.148846,-0.433565,-1.088380,-0.039820,-0.157395,-0.033712,-0.028018,-0.730598,-0.142174,-0.020069,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
4,-0.135680,0.107736,-0.039820,-0.033712,-0.157395,-0.028013,-0.823216,-0.098672,-0.433565,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,-0.051186,-0.289361,2.391576,-0.764465,0.196414,2.716388,0.134003,1.816166,-0.755174,0.199849,2.159151,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.055400,-0.048448,-0.286184,-0.069482,-0.143000,-1.041222,-1.150816,-1.083718,-0.830890,-0.381316,-0.141374,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,-0.557051,-1.148846,-0.433565,-1.088380,-0.039820,-0.157395,-0.033712,-0.028018,-0.719193,-0.142174,-0.020069,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,0.230399,-0.186839,0.205875,0.248680


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66
0,-0.13568,-0.253535,-0.039820,-0.033712,-0.157395,-0.028013,-0.823216,-0.098672,-0.433565,-0.856384,-1.023489,-0.176204,-1.088380,-0.997415,-0.051186,-0.289361,1.434935,-0.764465,-0.168358,1.731943,-0.221147,1.015533,-0.755174,-0.163694,1.350799,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,-0.141374,-0.055400,-0.048448,-0.286184,-0.069482,-0.143000,-1.041222,-1.150816,-1.083718,-0.830890,-0.381316,-0.141374,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,-0.557051,-1.148846,-0.433565,-1.088380,-0.039820,-0.157395,-0.033712,-0.028018,-0.728231,-0.142174,-0.020069,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,-0.132218,-0.186839,-0.155795,-0.111478
1,-0.13568,-0.704146,-0.000739,-0.010060,-0.004505,0.002136,0.422928,-0.098672,-0.034794,0.401023,2.485324,-0.176204,1.895768,2.971404,-0.051161,-0.289358,-0.411543,-0.586873,-0.739796,-0.243300,-0.777413,-0.532163,-0.602916,-0.733144,-0.271120,-0.094596,0.055353,-0.164177,-0.202201,-0.073737,-0.141374,0.006250,-0.008945,-0.286183,-0.069469,-0.143000,2.493365,1.265927,2.401873,3.803547,-0.381316,-0.141374,-0.0046,1.346322,-0.953045,-0.155378,-0.0046,-0.557051,1.266820,-0.034794,1.895768,-0.000739,-0.004505,-0.010060,0.002139,1.345897,-0.077680,-0.011800,0.432835,-0.267139,-0.07828,-0.269296,-0.256402,-0.700278,-0.186839,-0.722373,-0.675686
2,-0.13568,-0.980398,-0.047636,-0.027799,0.043877,-0.027998,0.817277,4.100901,3.241184,-0.856384,-1.021679,0.032926,-1.082223,-0.997415,0.086081,-0.224125,-0.484513,-0.764465,-0.900255,-0.243294,-0.933740,-0.590899,-0.755174,-0.893125,-0.271120,-0.470702,-0.387110,-0.407195,-0.427716,-0.073937,7.073435,-0.067143,-0.042639,-0.253292,0.337731,0.089607,-0.883211,-0.310070,-0.774896,-0.794449,-0.381316,7.073435,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,1.349835,0.006665,3.241184,-1.082223,-0.047636,0.043877,-0.027799,-0.028002,-0.732176,-0.141901,-0.020069,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
3,-0.13568,-0.758195,0.007078,0.001767,-0.024633,0.002136,0.258879,-0.098672,-0.130575,0.164972,0.723678,-0.176204,0.901052,0.739229,-0.051155,-0.289357,-0.439366,-0.638862,-0.782554,-0.243298,-0.830423,-0.557626,-0.662031,-0.787413,-0.271117,-0.167569,-0.173572,-0.227561,-0.262233,-0.073923,-0.141374,0.017993,0.009645,-0.286182,-0.069458,-0.143000,0.718779,0.774219,0.792164,0.513666,-0.381316,-0.141374,-0.0046,1.346322,-0.953045,-0.155378,-0.0046,-0.557051,0.740971,-0.130575,0.901052,0.007078,-0.024633,0.001767,0.002139,1.345897,-0.077680,-0.011800,0.432835,-0.267617,-0.07828,-0.269747,-0.256889,-0.859791,-0.186839,-0.881470,-0.834117
4,-0.13568,1.369783,0.014894,0.001767,-0.000635,0.002040,0.454476,-0.098672,-0.115558,0.280619,0.723678,-0.176204,0.894724,0.490898,-0.051183,-0.289361,-0.041070,1.118154,1.453115,-0.243300,1.357555,0.059938,1.185981,1.452313,-0.271120,-0.465965,-0.383772,-0.403041,-0.423173,-0.073933,-0.141374,0.029736,0.009645,-0.286184,-0.069480,-0.143000,0.718779,0.657672,0.558521,0.199592,-0.381316,-0.141374,-0.0046,-0.742765,1.049269,-0.155378,-0.0046,-0.557051,0.618665,-0.115558,0.894724,0.014894,-0.000635,0.001767,0.002043,-0.730598,-0.077680,-0.011800,0.432835,-0.266344,-0.07828,-0.268544,-0.255591,1.479671,-0.186839,1.451888,1.489481


## Model Selection and Training

## Classifier 1: Perceptron

In [32]:
# Construct a Perceptron classifier with learning rate of 0.001
pcp = Perceptron(eta0=0.001, random_state=1)

# train the model using the training dataset
# Fit the classifier with the training dataset
pcp.fit(X_train, y_train.flatten())

# Predict the labels for the test dataset
# Predict labels using the test dataset constructed from the features
pcp_preds = pcp.predict(X_test)

ValueError: Found input variables with inconsistent numbers of samples: [47257, 47346]

## Classifier 2: Logistic Regression

In [ ]:
# Construct a LogisticRegression classifier
lrg = LogisticRegression()

# train the model using the training dataset
# Fit the classifier with the training dataset
lrg.fit(X_train, y_train.flatten())

# Predict the labels for the test dataset
# Predict labels using the test dataset constructed from the features
lrg_preds = lrg.predict(X_test)


## Classifier 3: XGBoost

In [ ]:
# Construct a LogisticRegression classifier with 150 estimators
xgb = XGBClassifier(n_estimators=150)

# train the model using the training dataset
# Fit the classifier with the training dataset
xgb.fit(X_train, y_train.flatten())

# Predict the labels for the test dataset
# Predict labels using the test dataset constructed from the features
xgb_preds = xgb.predict(X_test)

## Model Evaluation

## Perceptron Results

In [ ]:
# compute the accuracy, display the computed measure
pcp_acc = accuracy_score(y_test, pcp_preds)
print('Perceptron classifier accuracy: {:.4f}'.format(pcp_acc))

# compute the F1-score, display the computed measure
pcp_F1 = f1_score(y_test, pcp_preds)
print('Perceptron classifier F1-score: {:.4f}'.format(pcp_F1))

# compute the precision, display the computed measure
pcp_pcn = precision_score(y_test, pcp_preds)
print('Perceptron classifier precision: {:.4f}'.format(pcp_pcn))

# compute the recall, display the computed measure
pcp_rcl = recall_score(y_test, pcp_preds)
print('Perceptron classifier recall: {:.4f}'.format(pcp_rcl))

In [ ]:
# display the confusion matrix
ConfusionMatrixDisplay.from_estimator(pcp, X_test, y_test)

## Logistic Regression Results

In [ ]:
# compute the accuracy, display the computed measure
lrg_acc = accuracy_score(y_test, lrg_preds)
print('Logistic regression classifier accuracy: {:.4f}'.format(lrg_acc))

# compute the F1-score, display the computed measure
lrg_F1 = f1_score(y_test, lrg_preds)
print('Logistic regression classifier F1-score: {:.4f}'.format(lrg_F1))

# compute the precision, display the computed measure
lrg_pcn = precision_score(y_test, lrg_preds)
print('Logistic regression classifier precision: {:.4f}'.format(lrg_pcn))

# compute the recall, display the computed measure
lrg_rcl = recall_score(y_test, lrg_preds)
print('Logistic regression classifier recall: {:.4f}'.format(lrg_rcl))

In [ ]:
# display the confusion matrix
ConfusionMatrixDisplay.from_estimator(lrg, X_test, y_test)

## XGBoost Results

In [ ]:
# compute the accuracy, display the computed measure
xgb_acc = accuracy_score(y_test, xgb_preds)
print('XGBoost classifier accuracy: {:.4f}'.format(xgb_acc))

# compute the F1-score, display the computed measure
xgb_F1 = f1_score(y_test, xgb_preds)
print('XGBoost classifier F1-score: {:.4f}'.format(xgb_F1))

# compute the precision, display the computed measure
xgb_pcn = precision_score(y_test, xgb_preds)
print('XGBoost classifier precision: {:.4f}'.format(xgb_pcn))

# compute the recall, display the computed measure
xgb_rcl = recall_score(y_test, xgb_preds)
print('XGBoost classifier recall: {:.4f}'.format(xgb_rcl))

In [ ]:
# display the confusion matrix
ConfusionMatrixDisplay.from_estimator(xgb, X_test, y_test)